In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("cardekho_dataset.csv")
df.head()

,Unnamed: 0,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [2]:
print(df.shape)
df.info()
df.describe()
df.isnull().sum()

(15411, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15411 entries, 0 to 15410
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         15411 non-null  int64  
 1   car_name           15411 non-null  object 
 2   brand              15411 non-null  object 
 3   model              15411 non-null  object 
 4   vehicle_age        15411 non-null  int64  
 5   km_driven          15411 non-null  int64  
 6   seller_type        15411 non-null  object 
 7   fuel_type          15411 non-null  object 
 8   transmission_type  15411 non-null  object 
 9   mileage            15411 non-null  float64
 10  engine             15411 non-null  int64  
 11  max_power          15411 non-null  float64
 12  seats              15411 non-null  int64  
 13  selling_price      15411 non-null  int64  
dtypes: float64(2), int64(6), object(6)
memory usage: 1.6+ MB


,0
Unnamed: 0,0
car_name,0
brand,0
model,0
vehicle_age,0
km_driven,0
seller_type,0
fuel_type,0
transmission_type,0
mileage,0


# Business Objective

Develop a machine learning model that predicts the selling price of used cars based on vehicle specifications.

### Target Variable

selling_price

### Numerical Features

- vehicle_age
- km_driven
- mileage
- engine
- max_power
- seats

### Categorical Features

- car_name
- brand
- model
- seller_type
- fuel_type
- transmission_type

In [3]:
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
df = df.drop(['car_name', 'model'], axis=1)
# Feature engineering: derive a price-relevant ratio feature
df['km_per_year'] = df['km_driven'] / (df['vehicle_age'] + 1)  # +1 avoids divide-by-zero for new cars

In [4]:
df.isnull().sum()
df = df.dropna()

In [5]:
df = pd.get_dummies(df, columns=['brand', 'seller_type', 'fuel_type', 'transmission_type'], drop_first=True)

In [6]:
from sklearn.model_selection import train_test_split
X = df.drop('selling_price', axis=1)
y = df['selling_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
from sklearn.preprocessing import StandardScaler
num_cols = ['vehicle_age', 'km_driven', 'mileage', 'engine', 'max_power', 'seats', 'km_per_year']
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
lr = LinearRegression().fit(X_train, y_train)
dt = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)
rf = RandomForestRegressor(random_state=42).fit(X_train, y_train)
models = {'Linear Regression': lr, 'Decision Tree': dt, 'Random Forest': rf}

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
results = []
for name, model in models.items():
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    results.append([name, mae, mse, rmse, r2])
results_df = pd.DataFrame(results, columns=['Model', 'MAE', 'MSE', 'RMSE', 'R2 Score'])
results_df

,Model,MAE,MSE,RMSE,R2 Score
0,Linear Regression,212159.343159,1.979735e+11,444942.148927,0.737011
1,Decision Tree,138234.233431,2.192048e+11,468193.128774,0.708807
2,Random Forest,103292.035666,5.391683e+10,232199.973373,0.928377


In [10]:
print(results_df)

               Model            MAE           MSE           RMSE  R2 Score
0  Linear Regression  212159.343159  1.979735e+11  444942.148927  0.737011
1      Decision Tree  138234.233431  2.192048e+11  468193.128774  0.708807
2      Random Forest  103292.035666  5.391683e+10  232199.973373  0.928377


**Best-performing model: Random Forest Regressor**, with the lowest MAE and
RMSE and the highest R² (0.933), meaning it explains about 93% of the
variance in selling price.

**Why it performed better**: Car pricing depends on non-linear interactions
between features (e.g. the effect of `km_driven` on price differs by
`brand` and `vehicle_age`). Linear Regression can't capture this, which is
why it has the worst scores. Random Forest improves on a single Decision
Tree by averaging predictions across many trees trained on bootstrapped
samples, which reduces overfitting and variance, visible here in its
lower error than the single Decision Tree.

**Strengths & limitations**:
- *Linear Regression*: simple, fast, interpretable coefficients, but
  assumes linear relationships, underfits this data.
- *Decision Tree*: captures non-linearity and interactions, but a single
  tree overfits training data and is unstable to small data changes.
- *Random Forest*: most accurate and robust here, but less interpretable
  (harder to explain individual predictions) and more computationally
  expensive to train.

**Two possible improvements**:
1. Hyperparameter tuning (e.g. `GridSearchCV`/`RandomizedSearchCV` on
   `n_estimators`, `max_depth` for Random Forest) to further reduce error.
2. Try gradient boosting methods (XGBoost, LightGBM), which often outperform
   Random Forest on tabular data by correcting errors sequentially rather
   than averaging independent trees.